In [1]:
import pandas as pd
from tqdm import tqdm 
import scanpy as sc
import sys
import os 
import logging 
from omegaconf import OmegaConf, DictConfig
from hydra import initialize, compose
from scipy.spatial.distance import cdist
from sklearn.preprocessing import LabelEncoder
import torch
import numpy as np
from pathlib import Path 
import seaborn as sns
import matplotlib.pyplot as plt

import torch.nn.functional as F

from sc_exp_design.constants import DataFields
sys.path.insert(0, "../../../../../../shared_utils")
from experiment_utils import (
    get_forward_model,
    get_target_dict, 
    get_loss_fn
)

logger = logging.getLogger(__name__)

In [2]:
def uncertainty_quantification(
    forward_model,
    base_cfg,
    annotation, 
    paths, 
    constraints,
    X_candidates, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column):
    
    protocol_columns = base_cfg["annotation"]["protocol_columns"]

    ct_le = LabelEncoder()
    ct_values = target_prediction_model.train_data.adata.obs[cell_type_column].values
    ct_le.fit(ct_values)
    classes = ct_le.classes_.tolist()  # List of cell types in alphabetical order 
    print(classes)

    # Take cell type index 
    cell_type_index = classes.index(target_cell_types)
    ct_safe_string = target_cell_types.replace("/", ":") # cell type dir

    X_candidates = torch.from_numpy(X_candidates).float().to("cuda")
    X_candidates_add = X_candidates.unsqueeze(1).repeat(1, n_noise_samples, 1)  # no_candidates x n_noise_sample x candidate_dim
    condition_repr = next(iter(forward_model.forward_model.train_data.data.perturbation_covariates))
    batch_dict = {DataFields.PERTURBATION_DATA: {condition_repr: X_candidates_add}}

    forward_out = forward_model.predict(
        batch_dict,
        no_grad=True,
        fix_noise=False,
        num_time_steps=100
    )
    
    y_pred = forward_out["target_prediction_data"][cell_type_column].view(X_candidates_add.shape[0],
                                                                     n_populations,
                                                                     -1,
                                                                     len(classes))  
    y_pred_softmax = F.softmax(y_pred, dim=-1)

    # Take the mean proportions 
    y_pred_mean = y_pred.mean(2)  # no_candidates x no_populations x no_cell_types 
    y_pred_softmax = y_pred_softmax.mean(2)  # no_candidates x no_populations x no_cell_types 
    print(y_pred_softmax[:, :, 21].mean())

    # Collect cell type of interest 
    y_pred_ct_std = y_pred_softmax[..., cell_type_index].std(1).detach().cpu().numpy()  # standard deviation prob cell type of interest 
    # Calculate loss
    y_target_ct = np.zeros((1, len(classes)))
    y_target_ct[:, cell_type_index] = 1.0
    y_target_ct = torch.from_numpy(y_target_ct)
    y_target_ct = y_target_ct.unsqueeze(0).to("cuda")  # 1 x 1 x no_cell_types
    
    logger.info("Compute loss function")
    loss_fn = lambda pred, target: -torch.sum(target*torch.nn.functional.log_softmax(pred, dim=-1), dim=-1)
    with torch.no_grad():
        loss = loss_fn(y_pred_mean, y_target_ct)  # no_candidates x no_populations

    loss_std = loss.std(1).detach().cpu().numpy()  # no_candidates
    return loss_std

In [3]:
p252 = np.array([9.2, 2.2, 997.3, 1120.2, 0, 57.2, 0, 0, 0.1, 0.6, 20, 14, 0, 0, 15])
p253 = np.array([9.2, 2.2, 997.3, 1120.2, 0, 57.2, 0, 0, 0.1, 0.6, 20, 14, 0, 0, 1.5])
p254 = np.array([10, 2.5, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 15])
p255 = np.array([10, 2.5, 750, 560, 0, 10, 0, 0, 0, 0, 20, 14, 0, 0, 1.5])

In [4]:
p252 = np.log1p(p252)
p253 = np.log1p(p253)
p254 = np.log1p(p254)
p255 = np.log1p(p255)

p252 = p252[None,:]
p253 = p253[None,:]
p254 = p254[None,:]
p255 = p255[None,:]

In [ ]:
base_config_path = "../../../../../../inverse/loss_guidance/config/"
base_config_name = "run_inverse"
annotation = "bloodplus_loop3"
paths = "loop3_replicate"
constraints = "default_all_cols_loop3"
target_cell_types = "preProB"
n_noise_samples = 5000
n_populations = 10
cell_type_column = "cell_type" 

# Initialize paths 
with initialize(config_path=base_config_path, version_base=None):
    base_cfg = compose(
        config_name=base_config_name,
        overrides=[f"paths={paths}", 
                   f"annotation={annotation}", 
                   f"constraints={constraints}"])
base_cfg.loss.cell_type_column = cell_type_column

# Collect the forward model
forward_model, (_, target_prediction_model) = get_forward_model(base_cfg, logger=logger)

In [ ]:
uncertainty_quantification(
    forward_model, 
    base_cfg,
    annotation, 
    paths, 
    constraints,
    p252, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column)

In [ ]:
uncertainty_quantification(
    forward_model, 
    base_cfg,
    annotation, 
    paths, 
    constraints,
    p253, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column)

In [ ]:
uncertainty_quantification(
    forward_model, 
    base_cfg,
    annotation, 
    paths, 
    constraints,
    p254, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column)

In [ ]:
uncertainty_quantification(
    forward_model, 
    base_cfg,
    annotation, 
    paths, 
    constraints,
    p255, 
    target_cell_types, 
    n_noise_samples,
    n_populations, 
    cell_type_column)

## Percentages 

In [ ]:
adata_l3 = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop3/replicate/processed/adata_loop3_500k_residual_processed.h5ad")
adata_l2p5 = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop2/processed/adata_loop2p5_500k_residual_processed.h5ad")
adata_l2 = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop2/processed/adata_loop2_500k_residual_processed.h5ad")
adata_l1 = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop1/adata_500k.h5ad")

In [ ]:
adata_l3.obs = adata_l3.obs.loc[:, ["experiment_number", "cell_type"]]
adata_l2p5.obs = adata_l2p5.obs.loc[:, ["experiment_number", "cell_type"]]
adata_l2.obs = adata_l2.obs.loc[:, ["experiment_number", "cell_type"]]
adata_l1.obs = adata_l1.obs.loc[:, ["experiment_number", "cell_type"]]

In [ ]:
adata_tot = sc.concat([adata_l1, adata_l2, adata_l2p5, adata_l3], axis=0)

In [ ]:
experiment_to_proportions = {}

for exp in ["252", "253", "254", "255"]:
    adata_exp = adata_tot[adata_tot.obs.experiment_number==exp]
    # preprob_prop = (adata_exp.obs.cell_type.value_counts() / adata_exp.obs.cell_type.value_counts().sum())["preProB"]
    preprob_prop = adata_exp.obs.cell_type.value_counts()["preProB"]
    experiment_to_proportions[exp] = preprob_prop

In [ ]:
experiment_to_proportions

## Number of PreProB 

In [ ]:
preProB_counts = {"experiment number": [], "preProB_count": []}

for exp in adata_tot.obs.experiment_number.unique():
    adata_exp = adata_tot[adata_tot.obs.experiment_number==exp]
    # preprob_prop = (adata_exp.obs.cell_type.value_counts() / adata_exp.obs.cell_type.value_counts().sum())["preProB"]
    try:
        preprob_c = adata_exp.obs.cell_type.value_counts()["preProB"]
    except:
        preprob_c = 0
        
    preProB_counts["experiment number"].append(exp)
    preProB_counts["preProB_count"].append(preprob_c)

In [ ]:
preProB_counts = pd.DataFrame(preProB_counts)

In [ ]:
preProB_counts["preProB_count"].max()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- Data ---
counts = pd.to_numeric(preProB_counts["preProB_count"], errors="coerce").dropna()

experiment_to_proportions = {
    '252': 144,
    '253': 139,
    '254': 106,
    '255': 82,
}

threshold = np.quantile(counts, 0.75)

# --- Style ---
sns.set_style("ticks")
plt.rcParams["font.size"] = 10
highlight_color = "#c0654f"
line_color = "black"
base_color = "#4c4c4c"

fig, ax = plt.subplots(figsize=(3.6, 3.0))

# Plain KDE, clipped at 0
sns.kdeplot(counts, ax=ax, color=base_color, linewidth=1.8, fill=False,
            cut=0, clip=(0, counts.max()))

kde_x, kde_y = ax.lines[0].get_data()

# Soft fill under the entire curve so the x=0 edge reads as a closed
# area shape rather than a stark vertical line
ax.fill_between(kde_x, kde_y, color=base_color, alpha=0.06, zorder=0)

# Highlight the top-quantile region on top of that
ax.fill_between(
    kde_x, kde_y, where=(kde_x >= threshold),
    color=highlight_color, alpha=0.15, zorder=1
)
ax.axvline(threshold, color=base_color, linestyle=":", linewidth=1, alpha=0.6)
ax.text(threshold, ax.get_ylim()[1]*0.92, "Top 25%",
        ha="left", va="top", fontsize=9, color=base_color, alpha=0.8)

# Mark the four experiments as small dots on the x-axis
exp_vals = list(experiment_to_proportions.values())
ax.scatter(exp_vals, [0]*len(exp_vals), color=line_color, s=12, zorder=4,
           clip_on=False)

ax.set_xlabel("preProB count")
ax.set_ylabel("Density")
ax.set_xlim(-5, 200)
ax.set_ylim(0, ax.get_ylim()[1])
sns.despine(ax=ax)
plt.tight_layout()

plt.savefig("preProB_top_quantile.svg", dpi=300, bbox_inches="tight")
plt.savefig("preProB_top_quantile.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
preProB_counts